# 04 — Création du dataset des compétences explosées

## Objectif du notebook

Ce notebook prépare un livrable propre à partir du dataset final nettoyé :

- fichier d'entrée : `final_data_ai_jobs_clean_reduced_2020_2026.csv` ou `final_data_ai_jobs_clean_2020_2026.csv`
- fichier de sortie : `final_data_ai_jobs_skills_exploded.csv`

Le but est de transformer la colonne `skills`, qui contient plusieurs compétences dans une seule cellule, en un dataset où chaque ligne représente **une compétence associée à une offre**.

Exemple :

`python, sql, aws` devient :

| job_id | skill |
|---|---|
| offre_1 | python |
| offre_1 | sql |
| offre_1 | aws |

Ce fichier sera utilisé pour :

- l'analyse des compétences,
- le Web Content Mining,
- le Graph Mining,
- la prédiction des tendances,
- MongoDB,
- le dashboard.


## 1. Importation des bibliothèques

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [1]:
import pandas as pd
import numpy as np
import re
import warnings

warnings.filterwarnings('ignore')

## 2. Chargement du dataset final clean

On charge en priorité la version réduite créée dans le notebook de qualité. Si elle n'existe pas, on utilise la version complète.

In [3]:
from pathlib import Path

reduced_file = Path('final_data_ai_jobs_clean_reduced_2020_2026.csv')
full_file = Path('final_data_ai_jobs_clean_2020_2026.csv')

if reduced_file.exists():
    input_file = reduced_file
else:
    input_file = full_file

print('Fichier utilisé :', input_file)

df = pd.read_csv(input_file)

print('Nombre de lignes :', df.shape[0])
print('Nombre de colonnes :', df.shape[1])
print(df.columns.tolist())

df.head(3).T

Fichier utilisé : final_data_ai_jobs_clean_reduced_2020_2026.csv
Nombre de lignes : 751801
Nombre de colonnes : 23
['source', 'platform', 'job_id', 'job_title', 'country', 'city', 'date_posted', 'year', 'month', 'year_month', 'remote_status', 'experience_level', 'education_required', 'category', 'skills', 'tools_used', 'original_file', 'country_clean', 'skills_clean', 'skills_original', 'remote_status_clean', 'remote_status_original', 'job_category_clean']


,0,1,2
source,global_ai_jobs_dataset,global_ai_jobs_dataset,global_ai_jobs_dataset
platform,Synthetic / Global,Synthetic / Global,Synthetic / Global
job_id,1,2,3
job_title,AI Researcher,MLOps Engineer,Data Analyst
country,Canada,India,United Kingdom
city,Berlin,Tokyo,Bangalore
date_posted,2021-04-01,2020-04-12,2023-01-31
year,2021,2020,2023
month,4.0,4.0,1.0
year_month,2021-04,2020-04,2023-01


## 3. Vérification de la colonne `skills`

Avant de créer le dataset des compétences, on vérifie que la colonne `skills` existe et qu'elle est suffisamment remplie.

In [5]:
if 'skills' not in df.columns:
    raise ValueError("La colonne 'skills' est absente du dataset.")

print("Nombre total d'offres :", df.shape[0])
print("Offres avec skills :", df['skills'].notna().sum())
print("Offres sans skills :", df['skills'].isna().sum())
print("Pourcentage avec skills :", round(df['skills'].notna().mean() * 100, 2), "%")

df[['job_id', 'job_title', 'country', 'year', 'skills']].head(10)

Nombre total d'offres : 751801
Offres avec skills : 751476
Offres sans skills : 325
Pourcentage avec skills : 99.96 %


,job_id,job_title,country,year,skills
0,1,AI Researcher,Canada,2021,"python, computer vision, sql, nlp"
1,2,MLOps Engineer,India,2020,"nlp, pytorch, data analysis, computer vision"
2,3,Data Analyst,United Kingdom,2023,"nlp, computer vision, python, pytorch"
3,4,NLP Engineer,Brazil,2022,"data analysis, statistics, tensorflow, python"
4,5,AI Researcher,Netherlands,2022,"nlp, machine learning, pytorch, sql"
5,6,Prompt Engineer,Brazil,2021,"tensorflow, computer vision, nlp, machine lear..."
6,7,AI Engineer,Singapore,2021,"statistics, python, tensorflow, machine learning"
7,8,AI Researcher,Brazil,2020,"computer vision, deep learning, tensorflow, py..."
8,9,Data Analyst,United States,2024,"statistics, machine learning, computer vision,..."
9,10,NLP Engineer,Singapore,2020,"deep learning, machine learning, sql, pytorch"


## 4. Nettoyage final des compétences

Les compétences peuvent provenir de plusieurs sources et avoir des formats différents :

- `python, sql, aws`
- `['python', 'sql', 'aws']`
- `Python; SQL; AWS`
- `python | sql | aws`

On applique donc une fonction de nettoyage avant l'explosion.

In [6]:
def clean_skills_text(value):
    if pd.isna(value):
        return np.nan
    
    text = str(value).lower().strip()
    
    # Suppression des caractères de listes/dictionnaires
    text = text.replace('[', ' ').replace(']', ' ')
    text = text.replace('{', ' ').replace('}', ' ')
    text = text.replace("'", ' ').replace('"', ' ')
    
    # Uniformisation des séparateurs
    text = text.replace(';', ',')
    text = text.replace('|', ',')
    
    # Attention : on ne remplace pas toujours '/' car certains termes comme ci/cd sont utiles.
    # On garde donc la version avec slash.
    
    # Nettoyage des espaces
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s*,\s*', ', ', text)
    
    skills = [s.strip() for s in text.split(',') if s.strip()]
    
    # Suppression des valeurs inutiles
    useless_values = {'nan', 'none', 'null', '[]', '{}', 'not specified', 'non spécifié'}
    skills = [s for s in skills if s not in useless_values]
    
    # Suppression des doublons en gardant l'ordre
    seen = set()
    cleaned = []
    for skill in skills:
        if skill not in seen:
            cleaned.append(skill)
            seen.add(skill)
    
    if len(cleaned) == 0:
        return np.nan
    
    return ', '.join(cleaned)

# Création d'une colonne propre pour les skills
df['skills_clean_final'] = df['skills'].apply(clean_skills_text)

print('Skills manquantes avant nettoyage :', df['skills'].isna().sum())
print('Skills manquantes après nettoyage :', df['skills_clean_final'].isna().sum())

df[['skills', 'skills_clean_final']].head(10)

Skills manquantes avant nettoyage : 325
Skills manquantes après nettoyage : 325


,skills,skills_clean_final
0,"python, computer vision, sql, nlp","python, computer vision, sql, nlp"
1,"nlp, pytorch, data analysis, computer vision","nlp, pytorch, data analysis, computer vision"
2,"nlp, computer vision, python, pytorch","nlp, computer vision, python, pytorch"
3,"data analysis, statistics, tensorflow, python","data analysis, statistics, tensorflow, python"
4,"nlp, machine learning, pytorch, sql","nlp, machine learning, pytorch, sql"
5,"tensorflow, computer vision, nlp, machine lear...","tensorflow, computer vision, nlp, machine lear..."
6,"statistics, python, tensorflow, machine learning","statistics, python, tensorflow, machine learning"
7,"computer vision, deep learning, tensorflow, py...","computer vision, deep learning, tensorflow, py..."
8,"statistics, machine learning, computer vision,...","statistics, machine learning, computer vision,..."
9,"deep learning, machine learning, sql, pytorch","deep learning, machine learning, sql, pytorch"


## 5. Création du dataset `skills_exploded`

On transforme chaque compétence en une ligne indépendante.

In [7]:
# Garder seulement les offres qui ont des skills exploitables
df_skills = df[df['skills_clean_final'].notna()].copy()

# Séparer les skills
df_skills['skill'] = df_skills['skills_clean_final'].str.split(',')

# Exploser les listes de skills
df_skills = df_skills.explode('skill')

# Nettoyer la compétence finale
df_skills['skill'] = (
    df_skills['skill']
    .astype(str)
    .str.strip()
    .str.lower()
)

# Supprimer les lignes vides
bad_values = ['', 'nan', 'none', 'null', '[]', '{}', 'not specified', 'non spécifié']
df_skills = df_skills[~df_skills['skill'].isin(bad_values)].copy()

print('Shape du dataset skills exploded :', df_skills.shape)
df_skills[['job_id', 'job_title', 'country', 'year', 'skill']].head(20)

Shape du dataset skills exploded : (3991435, 25)


,job_id,job_title,country,year,skill
0,1,AI Researcher,Canada,2021,python
0,1,AI Researcher,Canada,2021,computer vision
0,1,AI Researcher,Canada,2021,sql
0,1,AI Researcher,Canada,2021,nlp
1,2,MLOps Engineer,India,2020,nlp
1,2,MLOps Engineer,India,2020,pytorch
1,2,MLOps Engineer,India,2020,data analysis
1,2,MLOps Engineer,India,2020,computer vision
2,3,Data Analyst,United Kingdom,2023,nlp
2,3,Data Analyst,United Kingdom,2023,computer vision


## 6. Sélection des colonnes utiles

On garde uniquement les colonnes nécessaires pour l'analyse des compétences, le Graph Mining, la prédiction, MongoDB et le dashboard.

In [8]:
# Colonnes souhaitées pour le fichier skills exploded
candidate_columns = [
    'job_id',
    'source',
    'platform',
    'job_title',
    'country',
    'city',
    'date_posted',
    'year',
    'month',
    'year_month',
    'remote_status',
    'experience_level',
    'category',
    'tools_used',
    'salary',
    'original_file',
    'skill'
]

# Garder seulement les colonnes qui existent dans le dataframe
skills_columns = [col for col in candidate_columns if col in df_skills.columns]

print('Colonnes conservées :')
print(skills_columns)

df_skills_final = df_skills[skills_columns].copy()

print('Shape final skills :', df_skills_final.shape)
df_skills_final.head()

Colonnes conservées :
['job_id', 'source', 'platform', 'job_title', 'country', 'city', 'date_posted', 'year', 'month', 'year_month', 'remote_status', 'experience_level', 'category', 'tools_used', 'original_file', 'skill']
Shape final skills : (3991435, 16)


,job_id,source,platform,job_title,country,city,date_posted,year,month,year_month,remote_status,experience_level,category,tools_used,original_file,skill
0,1,global_ai_jobs_dataset,Synthetic / Global,AI Researcher,Canada,Berlin,2021-04-01,2021,4.0,2021-04,Remote,Senior,Generative AI,Kubernetes;PyTorch;AWS,global_ai_jobs_dataset.csv,python
0,1,global_ai_jobs_dataset,Synthetic / Global,AI Researcher,Canada,Berlin,2021-04-01,2021,4.0,2021-04,Remote,Senior,Generative AI,Kubernetes;PyTorch;AWS,global_ai_jobs_dataset.csv,computer vision
0,1,global_ai_jobs_dataset,Synthetic / Global,AI Researcher,Canada,Berlin,2021-04-01,2021,4.0,2021-04,Remote,Senior,Generative AI,Kubernetes;PyTorch;AWS,global_ai_jobs_dataset.csv,sql
0,1,global_ai_jobs_dataset,Synthetic / Global,AI Researcher,Canada,Berlin,2021-04-01,2021,4.0,2021-04,Remote,Senior,Generative AI,Kubernetes;PyTorch;AWS,global_ai_jobs_dataset.csv,nlp
1,2,global_ai_jobs_dataset,Synthetic / Global,MLOps Engineer,India,Tokyo,2020-04-12,2020,4.0,2020-04,Remote,Entry,Recommendation Systems,Python;Spark;Docker,global_ai_jobs_dataset.csv,nlp


## 7. Vérifications globales du dataset des skills

In [10]:
print('Nombre total de lignes skills :', df_skills_final.shape[0])
print('Nombre de colonnes :', df_skills_final.shape[1])
print('Nombre de skills différents :', df_skills_final['skill'].nunique())

print('Top 30 skills :')
print(df_skills_final['skill'].value_counts().head(30))

print('Lignes skills par année :')
print(df_skills_final.groupby('year')['skill'].count())

print('Top pays dans le dataset skills :')
print(df_skills_final['country'].value_counts(dropna=False).head(20))

Nombre total de lignes skills : 3991435
Nombre de colonnes : 16
Nombre de skills différents : 28277
Top 30 skills :
skill
sql                 423016
python              416081
aws                 133924
r                   131196
tableau             126609
excel               124313
azure               123904
spark               112002
power bi             97363
tensorflow           81587
java                 75943
pytorch              75727
hadoop               63677
scala                55324
snowflake            54116
databricks           52207
machine learning     51441
data analysis        50660
deep learning        48987
statistics           48874
computer vision      48523
nlp                  48231
gcp                  46906
kafka                46365
git                  45371
airflow              44073
nosql                42661
oracle               42144
sas                  41396
sql server           38240
Name: count, dtype: int64
Lignes skills par année :
year
2020      8

## 8. Vérification du Maroc

In [12]:
df_skills_morocco = df_skills_final[df_skills_final['country'] == 'Morocco']

print('Nombre de lignes skills Maroc :', df_skills_morocco.shape[0])
print('Nombre de skills différents au Maroc :', df_skills_morocco['skill'].nunique())

print('Top skills Maroc :')
print(df_skills_morocco['skill'].value_counts().head(30))

Nombre de lignes skills Maroc : 6690
Nombre de skills différents au Maroc : 206
Top skills Maroc :
skill
python        515
sql           510
spark         265
r             211
azure         209
tableau       193
power bi      191
java          181
hadoop        178
excel         172
aws           169
gcp           144
scala         123
kafka         115
docker        113
git            95
mysql          84
postgresql     81
jira           79
linux          78
kubernetes     75
nosql          75
databricks     74
oracle         73
sql server     72
jenkins        71
cloud          65
mongodb        61
ssis           60
sas            59
Name: count, dtype: int64


## 9. Sauvegarde du fichier final

In [14]:
output_file = 'final_data_ai_jobs_skills_exploded.csv'

df_skills_final.to_csv(output_file, index=False, encoding='utf-8-sig')

print('Fichier sauvegardé :', output_file)
print('Shape :', df_skills_final.shape)

Fichier sauvegardé : final_data_ai_jobs_skills_exploded.csv
Shape : (3991435, 16)


## 10. Test de rechargement

On recharge le fichier sauvegardé pour vérifier qu'il est correctement créé.

In [15]:
df_test = pd.read_csv('final_data_ai_jobs_skills_exploded.csv')

print('Shape après rechargement :', df_test.shape)
print(df_test.columns.tolist())

df_test.head()

Shape après rechargement : (3991435, 16)
['job_id', 'source', 'platform', 'job_title', 'country', 'city', 'date_posted', 'year', 'month', 'year_month', 'remote_status', 'experience_level', 'category', 'tools_used', 'original_file', 'skill']


,job_id,source,platform,job_title,country,city,date_posted,year,month,year_month,remote_status,experience_level,category,tools_used,original_file,skill
0,1,global_ai_jobs_dataset,Synthetic / Global,AI Researcher,Canada,Berlin,2021-04-01,2021,4.0,2021-04,Remote,Senior,Generative AI,Kubernetes;PyTorch;AWS,global_ai_jobs_dataset.csv,python
1,1,global_ai_jobs_dataset,Synthetic / Global,AI Researcher,Canada,Berlin,2021-04-01,2021,4.0,2021-04,Remote,Senior,Generative AI,Kubernetes;PyTorch;AWS,global_ai_jobs_dataset.csv,computer vision
2,1,global_ai_jobs_dataset,Synthetic / Global,AI Researcher,Canada,Berlin,2021-04-01,2021,4.0,2021-04,Remote,Senior,Generative AI,Kubernetes;PyTorch;AWS,global_ai_jobs_dataset.csv,sql
3,1,global_ai_jobs_dataset,Synthetic / Global,AI Researcher,Canada,Berlin,2021-04-01,2021,4.0,2021-04,Remote,Senior,Generative AI,Kubernetes;PyTorch;AWS,global_ai_jobs_dataset.csv,nlp
4,2,global_ai_jobs_dataset,Synthetic / Global,MLOps Engineer,India,Tokyo,2020-04-12,2020,4.0,2020-04,Remote,Entry,Recommendation Systems,Python;Spark;Docker,global_ai_jobs_dataset.csv,nlp


## Conclusion

Le dataset `final_data_ai_jobs_skills_exploded.csv` est maintenant prêt. Il contient une ligne par compétence et par offre.

Ce fichier sera utilisé pour :

- l'analyse des compétences les plus demandées,
- l'analyse des compétences par pays et par année,
- le Web Structure Mining avec le graphe de co-occurrence,
- le PageRank et Louvain,
- l'analyse prédictive des tendances de compétences,
- le stockage MongoDB dans la collection `skills_exploded`,
- le dashboard final.
